# Clase 162 — Policy gradients (REINFORCE)

**REINFORCE** (Williams, 1992) parametriza la policy con una red `π_θ(a|s)` y optimiza
directamente el expected return por gradiente:
`∇θ J = E[∇θ log π_θ(a|s) · G_t]`.

Requiere: `numpy`; `tensorflow`/`keras` opcional. El cálculo de returns/advantage va en numpy
(ejecutable); la red y el paso de gradiente usan Keras (no se ejecutan si TF no está instalado).

## 1. La red de policy: state → softmax(actions)

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import tensorflow as tf
    from tensorflow import keras
    tf.random.set_seed(42)
    TF_OK = True
    print("tensorflow", tf.__version__)
except Exception:
    TF_OK = False
    print("tensorflow no instalado -> se muestra la API de Keras (no se ejecuta)")

if TF_OK:
    policy = keras.Sequential([
        keras.Input(shape=(4,)),                       # state de CartPole
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(2, activation="softmax"),   # prob por accion
    ])
    policy.summary()
else:
    print("policy = Dense(32,relu) -> Dense(32,relu) -> Dense(2, softmax)")

## 2. Returns descontados y normalización (ejecutable)

Para cada timestep calculamos `G_t = Σ γ^k r_{t+k}`. Normalizar los returns (media 0, std 1)
reduce la varianza del gradiente y estabiliza el entrenamiento.

In [ ]:
def discounted_returns(rewards, gamma=0.99):
    G = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

rewards = np.ones(20)                       # un episodio de 20 pasos en CartPole
G = discounted_returns(rewards, gamma=0.99)
G_norm = (G - G.mean()) / (G.std() + 1e-8)
print("G[:4]     =", np.round(G[:4], 3))
print("G_norm[:4]=", np.round(G_norm[:4], 3))

## 3. El paso REINFORCE con `GradientTape`

`loss = -Σ log π_θ(a_t|s_t) · G_t`. Minimizarla equivale a subir por el gradiente de la
policy. El signo negativo convierte el *gradient ascent* en un *descent* estándar de Keras.

In [ ]:
if TF_OK:
    optimizer = keras.optimizers.Adam(learning_rate=1e-2)

    def reinforce_step(states, actions, returns):
        states  = tf.constant(states, dtype=tf.float32)
        actions = tf.constant(actions, dtype=tf.int32)
        returns = tf.constant(returns, dtype=tf.float32)
        with tf.GradientTape() as tape:
            probs = policy(states)                                  # (T, 2)
            idx = tf.stack([tf.range(tf.shape(actions)[0]), actions], axis=1)
            logp = tf.math.log(tf.gather_nd(probs, idx) + 1e-8)     # log pi(a_t|s_t)
            loss = -tf.reduce_mean(logp * returns)                  # REINFORCE
        grads = tape.gradient(loss, policy.trainable_variables)
        optimizer.apply_gradients(zip(grads, policy.trainable_variables))
        return float(loss)

    print("reinforce_step definido: log-prob x return, gradient ascent via Adam")
else:
    print("loss = -mean( log pi(a_t|s_t) * G_t ); grads = tape.gradient(loss, params)")

## 4. Baseline y advantage (ejecutable)

Restar un baseline `b(s)` (típicamente `V(s)`) del return reduce la varianza **sin introducir
sesgo**. El resultado es el **advantage** `A = G - V(s)`: cuánto mejor fue la acción que el
valor esperado del estado.

In [ ]:
V = np.full_like(G, G.mean())               # baseline constante = valor medio estimado
advantage = G - V
print("advantage[:4] =", np.round(advantage[:4], 3))
print("var(G)         =", round(float(G.var()), 3))
print("var(advantage) =", round(float(advantage.var()), 3), "(menor -> gradiente mas estable)")

## 5. El loop de entrenamiento (esquema)

```
for episodio in range(N):
    estados, acciones, recompensas = rollout(policy, env)   # 1 episodio completo
    G = discounted_returns(recompensas)
    A = (G - G.mean()) / (G.std() + 1e-8)                    # normalizar / baseline
    reinforce_step(estados, acciones, A)                     # 1 paso de gradiente
```

REINFORCE es **on-policy**: cada update usa datos generados por la policy actual. Su debilidad
es la alta varianza y la lentitud, lo que motiva A2C/PPO (clase 165).

## 6. Bonus de entropía para exploración (ejecutable)

Agregar `-β·H(π)` a la loss promueve políticas no determinísticas al inicio (más exploración).
La entropía `H(π) = -Σ π·log π` es máxima cuando todas las acciones son equiprobables.

In [ ]:
def entropy(probs):
    probs = np.clip(probs, 1e-8, 1.0)
    return -np.sum(probs * np.log(probs), axis=-1)

uniforme    = np.array([0.5, 0.5])
determinista = np.array([0.99, 0.01])
print(f"H(uniforme)    = {entropy(uniforme):.3f}  (maxima exploracion)")
print(f"H(determinista)= {entropy(determinista):.3f}  (poca exploracion)")
beta = 0.01
print(f"termino a restar de la loss: -beta*H = {-beta*entropy(uniforme):.4f}")

## Ejercicios

1. Implementar `rollout(policy, env)` que ejecute un episodio de CartPole y guarde
   `(estado, accion, recompensa)` por timestep.
2. Entrenar REINFORCE 500 episodios y graficar el return medio por época.
3. Añadir una cabeza `V(s)` (critic) y restar el baseline; comparar convergencia con/sin baseline.
4. Agregar un bonus de entropía `-β·H(π)` a la loss y observar el efecto en la exploración.

## Conclusiones

- REINFORCE optimiza la policy directamente: `∇θ J = E[∇θ log π · G_t]`.
- Se implementa con `GradientTape`: `loss = -mean(log π(a|s) · G)`.
- Normalizar los returns y usar un baseline reduce la varianza sin sesgar el gradiente.
- Es on-policy y de alta varianza; es la base conceptual de A2C, PPO y RLHF.